# Cambodian ALPR — rotation-training the PROVINCE classifier

## Why

The province classifier reads the Khmer line as a shape. It was never trained on
upside-down text, so it collapses when a plate is inverted. Measured 2026-08-23 on
the 567 held-out province crops:

| model | upright | upside-down |
|---|---|---|
| `province_classifier_best.pth` (deployed) | **96.1%** | **19.4%** |

A first local run with `--rotate180` for only 30 epochs reached **81.8% upright /
82.4% upside-down** — upside-down solved, but upright badly damaged, and the
validation curve was still climbing when it stopped. It had not converged.
This run gives it 100 epochs on a T4 to see whether upright recovers.

## Honest context — read this before you get excited

End-to-end, an upside-down plate is currently lost at the **detector**, not the
classifier: on inverted scenes the detector finds only **44 of 149** plates
(30%, vs 96% upright). So this improvement cannot show up in the full system
until the detector problem is solved separately.

This run is worth doing because it (a) lifts the 44 plates that ARE found, and
(b) means the province half is ready if the detector is ever fixed. It is not,
by itself, going to make your system read upside-down plates.

## What "good" looks like

Beat **96.1% upright / 19.4% upside-down**. Realistically upright will give up a
little to buy upside-down; the question is how much. If upright lands below ~92%
the trade is probably not worth deploying.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
import os, glob

tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
ok_flag = '--rotate180' in tr and 'rotate180: bool' in tr
ok_tool = os.path.exists('scripts/tools/test_province_rotation.py')
n_train = len(glob.glob('data/province_crops/train/*/*.jpg'))
n_test  = len(glob.glob('data/province_crops/test/*/*.jpg'))

print('--rotate180 flag      :', 'PRESENT' if ok_flag else '*** MISSING ***')
print('test_province_rotation:', 'PRESENT' if ok_tool else '*** MISSING ***')
print('province crops        : train', n_train, '| test', n_test)

assert ok_flag and ok_tool, 'STALE BUNDLE -- rebuild with --crnn-only and re-upload.'
assert n_train >= 2500 and n_test >= 560, 'province crops missing from the bundle'
print()
print('Bundle is current. Safe to continue.')

In [ ]:
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless

In [ ]:
# BASELINE -- the deployed classifier, both orientations. Expect 96.1% / 19.4%.
!python scripts/tools/test_province_rotation.py \
    --weights models/recognition/province_classifier_best.pth

In [ ]:
# THE EXPERIMENT -- 100 epochs with exact-180 flip augmentation (p=0.5).
# --rotate180 flips exactly upside-down; --rotate (the older flag) spreads the
# model over EVERY angle including 90 deg, where the square crop cuts off most of
# a wide Khmer line. Plates are upright or inverted, so train for those two.
!python scripts/recognition/train_province_classifier.py --rotate180 --epochs 100 \
    --out models/recognition/province_classifier_rot2.pth

In [ ]:
# MEASURE -- the honest verdict, same 567 crops as the baseline above.
!python scripts/tools/test_province_rotation.py \
    --weights models/recognition/province_classifier_rot2.pth \
    --config models/recognition/province_classifier_rot2_config.json

### How to read it

Compare against **96.1% upright / 19.4% upside-down**:

| what you see | what it means | what to do |
|---|---|---|
| upright ≥ ~94%, upside-down ≥ ~85% | the trade is cheap — a clear win | promote it |
| upright 92–94%, upside-down high | acceptable trade, your call | probably promote |
| upright < 92% | too costly; one model cannot do both here | keep `best.pth`; consider two models, picked by orientation |
| upright still ~82% like the 30-epoch run | more epochs did not help — capacity, not training time | tell Claude |

Remember the detector caveat above: even a perfect result here does not fix
end-to-end upside-down reading on its own.

Paste the output of the last three cells back to Claude in full.

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
saved = 0
for f in ('province_classifier_rot2.pth', 'province_classifier_rot2_config.json'):
    p = 'models/recognition/' + f
    if os.path.exists(p):
        shutil.copy(p, '/content/drive/MyDrive/ALPR/trained/')
        print('saved', f)
        saved += 1
if saved < 2:
    print('SOMETHING MISSING -- training may have failed, check the cell above.')
print('Both the .pth AND the _config.json are needed on the laptop.')